# Downscaling validation for air and dew point temperature

In [ ]:
import datetime as dt
import os

import pandas as pd


In [ ]:
from etdataset import icos

In [ ]:
from etdataset.logging import LoggerManager

logger = LoggerManager.get_logger(__name__)


### ICOS stations

Get a csv file with all the valid ICOS stations and their metadata

In [ ]:
path_icos_stat = icos.get_csv_with_valid_icos_stations()


Check the available stations by country number

In [ ]:
icos.filter_stations_by_country_code(path_icos_stat, "FR")

In [ ]:
# API token from your ICOS account
auth_token = os.getenv("ICOS_API_TOKEN")

In [ ]:
icos.download_icos_station(auth_token, ["DE-Geb", "FR-CLt"])

In [ ]:
cfg = icos.load_stations_config(path_icos_stat)

### Retrieve the time series for ICOS, ERA5, and downscaled ERA5 datasets.

In [ ]:
output = "out"
os.makedirs(output, exist_ok=True)

In [ ]:
start_date = dt.date(2023, 5, 3)
end_date = dt.date(2023, 5, 4)

Download ERA5 datasets date by date

In [ ]:
icos.download_date_by_date(
    date1=start_date,
    date2=end_date,
    output=output,
)


Generate time series for multiple stations in parallel using multiprocessing. 
The time series include ICOS measurements, ERA5 data, and downscaled ERA5 data, and are saved as CSV files.

In [ ]:
if __name__ == "__main__":
    path_icos_stat = "stations_meteoL2.csv"
    cfg = icos.load_stations_config(path_icos_stat)
    output = "out"
    os.makedirs(output, exist_ok=True)

    res = icos.generate_timeseries_for_stations_multiprocess(
        ["DE-Geb", "CH-Dav", "FR-CLt"],
        cfg,
        output,
        start_date,
        end_date,
        "csv",
    )


Read the csv files

In [ ]:
df_ge = pd.read_csv(
    "csv/DE-Geb_timeseries.csv",
    parse_dates=["time"],
)
df_dav = pd.read_csv(
    "csv/CH-Dav_timeseries.csv",
    parse_dates=["time"],
)
df_clt = pd.read_csv(
    "csv/FR-CLt_timeseries.csv",
    parse_dates=["time"],
)


stations = {
    "Geb": df_ge,
    "Dav": df_dav,
    "CLt": df_clt,
}


Plot air and dew point temperature by the selected months

In [ ]:
icos.plot_ta_tdp_csv(stations, "2023-03", "2023-04")

Metrics

In [ ]:
res_ge_rmse = icos.compute_monthly(
    "csv/DE-Geb_timeseries.csv",
    "rmse",  # or mae, r2, mbe or slope
)
res_ge_r2 = icos.compute_monthly(
    "csv/DE-Geb_timeseries.csv",
    "rmse",
)

res_ge_slope = icos.compute_monthly(
    "csv/DE-Geb_timeseries.csv", "slope", plot=True
)

In [ ]:
res_ge_rmse

In [ ]:
res_ge_r2

In [ ]:
res_ge_slope